# Interview Prep GraphRAG: Build Knowledge Graph

This notebook follows the same pattern as the GraphRAG reference notebook:

1. Load a pre-scraped article dataset
2. Define an ontology and extraction schema
3. Build a custom `GraphRAGExtractor`
4. Build a `PropertyGraphIndex`
5. Run community detection + community summaries
6. Export a deployable HTML graph
7. Answer role/topic questions with GraphRAG

**Input**
- `interview_prep_corpus.csv`

**Outputs**
- `interview_graph_data.json`
- `job_skill_graph.html`

---
## 0. Install Dependencies

In [1]:
%pip install -q llama-index llama-index-llms-openai graspologic pandas nest-asyncio python-dotenv networkx pydantic


Note: you may need to restart the kernel to use updated packages.


---
## 1. Imports & Configuration

In [2]:
import asyncio
import copy
import json
import pathlib
import re
from collections import Counter
from typing import List, Literal

import nest_asyncio
import networkx as nx
import pandas as pd
from dotenv import load_dotenv
from graspologic.partition import hierarchical_leiden
from pydantic import BaseModel, Field, field_validator

from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.async_utils import run_jobs
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.graph_stores.types import EntityNode, KG_NODES_KEY, KG_RELATIONS_KEY, Relation
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.schema import BaseNode, TransformComponent
from llama_index.llms.openai import OpenAI

nest_asyncio.apply()
load_dotenv()

print("✅ Imports ready")


c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports ready


---
## 2. Model Configuration

Use `gpt-4o-mini` for extraction and community summarization, and `gpt-4o` for final answer synthesis.

In [3]:
EXTRACTION_LLM = OpenAI(model="gpt-4o-mini", temperature=0)
SUMMARY_LLM    = OpenAI(model="gpt-4o-mini", temperature=0)
QUERY_LLM      = OpenAI(model="gpt-4o", temperature=0)

MAX_ARTICLES        = 60
MAX_PATHS_PER_CHUNK = 12
NUM_WORKERS         = 4
MAX_CLUSTER_SIZE    = 8
GRAPH_OUTPUT_FILE   = "job_skill_graph.html"

Settings.llm = EXTRACTION_LLM

print(f"✅ Using {EXTRACTION_LLM.model} for extraction")
print(f"✅ Using {SUMMARY_LLM.model} for summaries")
print(f"✅ Using {QUERY_LLM.model} for final query synthesis")


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Using gpt-4o-mini for extraction
✅ Using gpt-4o-mini for summaries
✅ Using gpt-4o for final query synthesis


---
## 3. Ontology Prompt

In [4]:
ENTITY_TYPES = [
    "JOB_ROLE",
    "INTERVIEW_TOPIC",
    "SKILL",
    "TOOL",
    "ALGORITHM",
    "RESOURCE",
]

RELATION_TYPES = [
    "TESTS",
    "REQUIRES",
    "USES",
    "RELATED_TO",
    "RECOMMENDED_FOR",
]

KG_TRIPLET_EXTRACT_TMPL = '''
You are extracting a knowledge graph from interview preparation articles.

Allowed entity types:
{entity_types}

Allowed relationship types:
{relation_types}

Instructions:
- Focus only on interview preparation content.
- Extract canonical job roles such as Machine Learning Engineer, AI Engineer, Data Scientist, MLOps Engineer, Data Engineer, NLP Engineer, and Computer Vision Engineer.
- INTERVIEW_TOPIC nodes should be interview-facing subjects like ML System Design, A/B Testing, Feature Engineering, Model Evaluation, SQL Optimization, Experimentation, Behavioral Interviews.
- SKILL nodes should be broad capabilities like Statistics, Probability, Linear Algebra, Python, Communication.
- TOOL nodes should be concrete tools or frameworks like PyTorch, TensorFlow, Spark, Docker, Airflow, Kubernetes.
- ALGORITHM nodes should be named methods like XGBoost, Random Forest, Transformers, Logistic Regression, CNNs.
- RESOURCE nodes should be books, platforms, courses, or study resources explicitly recommended by the article.
- Do not extract companies, salaries, locations, departments, benefits, or generic placeholder nouns.
- Every entity should have a short description.
- Every relationship should have a sentence-length description.
- Return at most {max_knowledge_triplets} high-quality relationship paths.

Article text:
---------------------
{text}
---------------------
'''.strip()

KG_TRIPLET_EXTRACT_TMPL = KG_TRIPLET_EXTRACT_TMPL.format(
    entity_types=", ".join(ENTITY_TYPES),
    relation_types=", ".join(RELATION_TYPES),
    max_knowledge_triplets="{max_knowledge_triplets}",
    text="{text}",
)


---
## 4. Pydantic Extraction Models

In [5]:
EntityTypeStr = Literal[
    "JOB_ROLE",
    "INTERVIEW_TOPIC",
    "SKILL",
    "TOOL",
    "ALGORITHM",
    "RESOURCE",
]

RelationTypeStr = Literal[
    "TESTS",
    "REQUIRES",
    "USES",
    "RELATED_TO",
    "RECOMMENDED_FOR",
]


class ExtractedEntity(BaseModel):
    name: str = Field(description="Name of the entity")
    type: EntityTypeStr = Field(description="Allowed entity type")
    description: str = Field(description="Brief description of the entity")


class ExtractedRelationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    relation: RelationTypeStr = Field(description="Allowed relationship type")
    description: str = Field(description="Sentence describing the relationship")


class ExtractionResult(BaseModel):
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)


print("✅ Pydantic extraction models defined")


✅ Pydantic extraction models defined


---
## 5. GraphRAGExtractor

In [6]:
class GraphRAGExtractor(TransformComponent):
    llm: LLM = Field(default_factory=lambda: Settings.llm)
    extract_prompt: PromptTemplate = Field(default_factory=lambda: PromptTemplate(KG_TRIPLET_EXTRACT_TMPL))
    num_workers: int = 4
    max_paths_per_chunk: int = 10

    @field_validator("extract_prompt", mode="before")
    @classmethod
    def coerce_to_prompt_template(cls, value):
        return PromptTemplate(value) if isinstance(value, str) else value

    def __call__(self, nodes, show_progress=False, **kwargs):
        return asyncio.run(self.acall(nodes, show_progress=show_progress, **kwargs))

    async def _aextract(self, node: BaseNode) -> BaseNode:
        text = node.get_content(metadata_mode="llm")

        try:
            result = await self.llm.astructured_predict(
                ExtractionResult,
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities = result.entities
            relationships = result.relationships
        except Exception as exc:
            print(f"Extraction error: {exc}")
            entities, relationships = [], []

        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        base_metadata = node.metadata.copy()

        existing_nodes += [
            EntityNode(
                name=entity.name,
                label=entity.type,
                properties={**base_metadata, "entity_description": entity.description},
            )
            for entity in entities
        ]

        entity_lookup = {entity.name: entity.type for entity in entities}

        for rel in relationships:
            source_node = EntityNode(
                name=rel.source,
                label=entity_lookup.get(rel.source, "JOB_ROLE"),
                properties=base_metadata,
            )
            target_node = EntityNode(
                name=rel.target,
                label=entity_lookup.get(rel.target, "INTERVIEW_TOPIC"),
                properties=base_metadata,
            )

            if rel.source not in entity_lookup:
                existing_nodes.append(source_node)
            if rel.target not in entity_lookup:
                existing_nodes.append(target_node)

            existing_relations.append(
                Relation(
                    label=rel.relation,
                    source_id=source_node.id,
                    target_id=target_node.id,
                    properties={**base_metadata, "relationship_description": rel.description},
                )
            )

        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(self, nodes, show_progress=False, **kwargs):
        jobs = [self._aextract(node) for node in nodes]
        return await run_jobs(jobs, workers=self.num_workers, show_progress=show_progress, desc="Extracting graph paths")


print("✅ GraphRAGExtractor ready")


✅ GraphRAGExtractor ready


---
## 6. GraphRAGStore

In [7]:
TYPE_PRIORITY = {
    "JOB_ROLE": 6,
    "SKILL": 5,
    "INTERVIEW_TOPIC": 4,
    "ALGORITHM": 4,
    "TOOL": 3,
    "RESOURCE": 2,
    "OTHER": 1,
}

RELATION_PRIORITY = {
    "REQUIRES": 6,
    "TESTS": 5,
    "USES": 4,
    "RELATED_TO": 3,
    "RECOMMENDED_FOR": 2,
}

CANONICAL_LABEL_OVERRIDES = {
    ("ALGORITHM", "yolo (you only look once)"): "YOLO",
    ("ALGORITHM", "cnn (convolutional neural network)"): "CNNs",
    ("ALGORITHM", "convolutional neural networks (cnn)"): "CNNs",
    ("ALGORITHM", "convolutional neural networks (cnns)"): "CNNs",
    ("INTERVIEW_TOPIC", "machine learning systems design"): "Machine Learning System Design",
    ("INTERVIEW_TOPIC", "named entity recognition (ner)"): "Named Entity Recognition",
    ("TOOL", "dvc (data version control)"): "DVC",
}


class GraphRAGStore(SimplePropertyGraphStore):
    community_summaries: dict = {}
    community_membership: dict = {}
    processed_stats: dict = {}

    def _canonical_label(self, label: str, node_type: str) -> str:
        cleaned = re.sub(r"\s+", " ", (label or "").strip())
        if not cleaned:
            return ""
        override = CANONICAL_LABEL_OVERRIDES.get((node_type or "OTHER", cleaned.lower()))
        return override or cleaned

    def _processed_graph_bundle(self):
        if hasattr(self, "_processed_graph_cache") and self._processed_graph_cache is not None:
            cached = self._processed_graph_cache
            return {
                "graph": cached["graph"].copy(),
                "stats": dict(cached["stats"]),
            }

        raw_nodes = {
            node.id: node
            for node in self.graph.nodes.values()
            if isinstance(node, EntityNode)
        }
        graph = nx.Graph()
        alias_map = {}

        for raw_id, node in raw_nodes.items():
            node_type = node.label if node.label else "OTHER"
            canonical_id = self._canonical_label(node.name, node_type)
            if not canonical_id:
                continue

            alias_map[raw_id] = canonical_id
            description = (node.properties.get("entity_description") or "").strip()

            if graph.has_node(canonical_id):
                attrs = graph.nodes[canonical_id]
                attrs["raw_ids"].add(raw_id)
                if TYPE_PRIORITY.get(node_type, 0) > TYPE_PRIORITY.get(attrs["type"], 0):
                    attrs["type"] = node_type
                if len(description) > len(attrs["description"]):
                    attrs["description"] = description
            else:
                graph.add_node(
                    canonical_id,
                    label=canonical_id,
                    type=node_type,
                    description=description,
                    raw_ids={raw_id},
                )

        for relation in self.graph.relations.values():
            source_node = raw_nodes.get(relation.source_id)
            target_node = raw_nodes.get(relation.target_id)
            if source_node is None or target_node is None:
                continue

            source_id = alias_map.get(relation.source_id)
            target_id = alias_map.get(relation.target_id)
            if not source_id or not target_id or source_id == target_id:
                continue

            rel_label = relation.label or "RELATED_TO"
            description = (relation.properties.get("relationship_description") or "").strip()

            if graph.has_edge(source_id, target_id):
                edge = graph[source_id][target_id]
                edge["weight"] += 1
                edge["relationship_counts"][rel_label] = edge["relationship_counts"].get(rel_label, 0) + 1
                if description and description not in edge["descriptions"]:
                    edge["descriptions"].append(description)
            else:
                graph.add_edge(
                    source_id,
                    target_id,
                    weight=1,
                    relationship=rel_label,
                    relationship_counts={rel_label: 1},
                    descriptions=[description] if description else [],
                )

        for _, _, edge in graph.edges(data=True):
            ranked = sorted(
                edge["relationship_counts"].items(),
                key=lambda item: (-item[1], -RELATION_PRIORITY.get(item[0], 0), item[0]),
            )
            edge["relationship"] = ranked[0][0] if ranked else "RELATED_TO"
            edge["description"] = edge["descriptions"][0] if edge["descriptions"] else ""

        raw_node_count = len(graph.nodes)
        raw_edge_count = len(graph.edges)

        removed_isolates = len(list(nx.isolates(graph)))
        if removed_isolates:
            graph.remove_nodes_from(list(nx.isolates(graph)))

        roleless_components = 0
        for component in list(nx.connected_components(graph)):
            if not any(graph.nodes[node_id].get("type") == "JOB_ROLE" for node_id in component):
                graph.remove_nodes_from(component)
                roleless_components += 1

        removed_leaf_resources = 0
        changed = True
        while changed:
            changed = False
            to_remove = []
            for node_id, attrs in graph.nodes(data=True):
                if attrs.get("type") != "RESOURCE":
                    continue
                if graph.degree(node_id) <= 1:
                    to_remove.append(node_id)
            if to_remove:
                graph.remove_nodes_from(to_remove)
                removed_leaf_resources += len(to_remove)
                changed = True

        trailing_isolates = len(list(nx.isolates(graph)))
        if trailing_isolates:
            graph.remove_nodes_from(list(nx.isolates(graph)))

        role_nodes = [node_id for node_id, attrs in graph.nodes(data=True) if attrs.get("type") == "JOB_ROLE"]
        role_distances = nx.multi_source_dijkstra_path_length(graph, role_nodes) if role_nodes else {}

        for node_id, attrs in graph.nodes(data=True):
            degree = graph.degree(node_id)
            attrs["degree"] = degree
            attrs["role_distance"] = role_distances.get(node_id)
            attrs["importance"] = (
                100 if attrs.get("type") == "JOB_ROLE"
                else degree * 8
                + (6 if attrs.get("role_distance") == 1 else 0)
                + (3 if attrs.get("type") in {"SKILL", "INTERVIEW_TOPIC"} else 0)
            )

        bundle = {
            "graph": graph.copy(),
            "stats": {
                "raw_nodes": raw_node_count,
                "raw_edges": raw_edge_count,
                "processed_nodes": graph.number_of_nodes(),
                "processed_edges": graph.number_of_edges(),
                "removed_initial_isolates": removed_isolates,
                "removed_roleless_components": roleless_components,
                "removed_leaf_resources": removed_leaf_resources,
                "removed_trailing_isolates": trailing_isolates,
            },
        }
        self._processed_graph_cache = {
            "graph": graph.copy(),
            "stats": dict(bundle["stats"]),
        }
        return bundle

    def get_processed_graph(self):
        bundle = self._processed_graph_bundle()
        return {
            "graph": bundle["graph"].copy(),
            "stats": dict(bundle["stats"]),
        }

    def build_communities(self):
        print("Running community detection...")
        bundle = self._processed_graph_bundle()
        nx_graph = bundle["graph"]
        self.processed_stats = dict(bundle["stats"])

        if not nx_graph.nodes:
            print("⚠️ Graph is empty — no communities to detect")
            self.community_summaries = {}
            self.community_membership = {}
            return

        print(
            f"Raw graph: {self.processed_stats['raw_nodes']} nodes, "
            f"{self.processed_stats['raw_edges']} edges"
        )
        print(
            f"Processed graph: {nx_graph.number_of_nodes()} nodes, "
            f"{nx_graph.number_of_edges()} edges"
        )
        print(
            "Removed "
            f"{self.processed_stats['removed_initial_isolates']} initial isolates, "
            f"{self.processed_stats['removed_leaf_resources']} low-signal resources, and "
            f"{self.processed_stats['removed_roleless_components']} roleless components"
        )

        clusters = hierarchical_leiden(nx_graph, max_cluster_size=MAX_CLUSTER_SIZE)
        self.community_membership = {item.node: item.cluster for item in clusters}
        print(f"Found {len(set(self.community_membership.values()))} communities")

        community_info = self._collect_community_info(nx_graph, clusters)
        self._generate_summaries(community_info)
        print(f"✅ Generated {len(self.community_summaries)} community summaries")

    def get_community_summaries(self):
        return self.community_summaries

    def _to_networkx(self) -> nx.Graph:
        return self.get_processed_graph()["graph"]

    def _collect_community_info(self, nx_graph, clusters):
        community_mapping = {item.node: item.cluster for item in clusters}
        community_info = {}

        for item in clusters:
            community_info.setdefault(item.cluster, {"entities": [], "relationships": []})
            node_id = item.node
            attrs = nx_graph.nodes[node_id]
            community_info[item.cluster]["entities"].append(
                {
                    "name": attrs.get("label", node_id),
                    "type": attrs.get("type", "OTHER"),
                    "description": attrs.get("description", ""),
                }
            )

        for source, target, edge in nx_graph.edges(data=True):
            cid = community_mapping.get(source)
            if cid is None or community_mapping.get(target) != cid:
                continue
            src_name = nx_graph.nodes[source].get("label", source)
            tgt_name = nx_graph.nodes[target].get("label", target)
            entry = f"{src_name} --[{edge.get('relationship', 'RELATED_TO')}]--> {tgt_name}"
            if edge.get("description"):
                entry += f" ({edge['description']})"
            community_info[cid]["relationships"].append(entry)

        return community_info

    def _generate_summaries(self, community_info):
        self.community_summaries = {}

        for community_id, data in community_info.items():
            entities_text = "\n".join(
                f"- {entity['name']} ({entity['type']}): {entity['description']}"
                for entity in data["entities"]
                if entity.get("name")
            )
            relationships_text = "\n".join(sorted(set(data["relationships"])))

            prompt = f'''You are analyzing a cluster from an interview preparation knowledge graph.

Entities:
{entities_text}

Relationships:
{relationships_text}

Write a concise 3-5 sentence briefing that:
1. Explains the main role, skill, topic, algorithm, tool, or resource cluster
2. Describes how the entities connect in interview preparation
3. Highlights what a candidate should study or understand in this area
'''

            response = SUMMARY_LLM.complete(prompt)
            self.community_summaries[community_id] = str(response)


print("✅ GraphRAGStore ready")


✅ GraphRAGStore ready


---
## 7. Load Article Dataset

In [8]:
df = pd.read_csv("interview_prep_corpus.csv")
df = df.loc[df["keep_for_graph"].fillna(False).astype(bool)].copy()
df = df.dropna(subset=["clean_text"]).copy()
df = df.head(MAX_ARTICLES).reset_index(drop=True)

print(f"Number of approved articles: {len(df)}")
df.head()


Number of approved articles: 56


,job_role_hint,query,title,snippet,source,href,domain,raw_text,clean_text,text_length_raw,text_length_clean,keyword_hits,content_type,keep_for_graph,drop_reason
0,AI Engineer,Applied AI Engineer interview questions,Common AI Engineer Interview Questions & Answe...,"Question 6: What is transfer learning, and how...",365 Data Science,https://365datascience.com/career-advice/job-i...,365datascience.com,It seems like every day we hear about some new...,It seems like every day we hear about some new...,20671,20671,8,question_bank,True,NaN
1,AI Engineer,AI Engineer interview preparation roadmap,alexeygrigorev/ai-engineering-field-guide,Interview Preparation · Interview process - co...,GitHub,https://github.com/alexeygrigorev/ai-engineeri...,github.com,Data-driven field guide to AI engineering role...,Data-driven field guide to AI engineering role...,4110,4110,7,system_design_guide,True,NaN
2,AI Engineer,Applied AI Engineer interview questions,amitshekhariitbhu/ai-engineering-interview-que...,"What is prompt engineering, and why is it crit...",GitHub,https://github.com/amitshekhariitbhu/ai-engine...,github.com,AI Engineering Interview Questions and Answers...,AI Engineering Interview Questions and Answers...,35049,35049,8,system_design_guide,True,NaN
3,AI Engineer,AI Engineer interview preparation roadmap,Interviewing for ML/AI Engineers,"For staff+ candidates, I would do a coding int...",Brian Kihoon Lee,https://www.moderndescartes.com/essays/ml_eng_...,moderndescartes.com,Interviewing for ML/AI Engineers\n2025-12-22\n...,Interviewing for ML/AI Engineers 2025-12-22 Ta...,15473,15473,8,system_design_guide,True,NaN
4,AI Engineer,AI Engineer interview preparation roadmap,How to pass a technical interview for an AI role,Think of the topic you need to explore as a ma...,Substack · Pau Labarta Bajo's Newsletter,https://paulabartabajo.substack.com/p/how-to-p...,substack.com,How to pass a technical interview for an AI ro...,How to pass a technical interview for an AI ro...,9091,9091,5,question_bank,True,NaN


---
## 8. Wrap Articles as Documents

In [9]:
nodes = [
    Document(
        text=str(row["clean_text"]),
        metadata={
            "title": str(row.get("title", "")),
            "source": str(row.get("source", "")),
            "query": str(row.get("query", "")),
            "job_role_hint": str(row.get("job_role_hint", "")),
            "url": str(row.get("href", "")),
            "content_type": str(row.get("content_type", "")),
        },
    )
    for _, row in df.iterrows()
]

print(f"✅ Created {len(nodes)} documents")


✅ Created 56 documents


---
## 9. Build the Knowledge Graph

In [10]:
kg_extractor = GraphRAGExtractor(
    llm=EXTRACTION_LLM,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
)

graph_store = GraphRAGStore()

print("Building interview-prep knowledge graph...")

index = PropertyGraphIndex(
    nodes=nodes,
    kg_extractors=[kg_extractor],
    property_graph_store=graph_store,
    embed_kg_nodes=False,
    show_progress=True,
)

print("✅ Knowledge graph built")


Building interview-prep knowledge graph...


Applying transformations: 100%|██████████| 1/1 [03:41<00:00, 221.91s/it]

✅ Knowledge graph built


---
## 10. Inspect One Extraction

In [11]:
def inspect_extraction(sample_index: int = 0):
    sample = nodes[sample_index]

    print("=== Title ===")
    print(sample.metadata.get("title", "untitled"))
    print()
    print("=== Raw text ===")
    print(sample.text[:2500])
    print()

    loop = asyncio.get_event_loop()
    extracted = loop.run_until_complete(kg_extractor._aextract(copy.deepcopy(sample)))

    print("=== Extracted entities ===")
    for node in extracted.metadata.get(KG_NODES_KEY, []):
        print(f"  [{node.label}] {node.name}")
        print(f"    {node.properties.get('entity_description', '')}")

    print()
    print("=== Extracted relationships ===")
    id_to_name = {n.id: n.name for n in extracted.metadata.get(KG_NODES_KEY, [])}
    for rel in extracted.metadata.get(KG_RELATIONS_KEY, []):
        src = id_to_name.get(rel.source_id, rel.source_id)
        tgt = id_to_name.get(rel.target_id, rel.target_id)
        print(f"  {src} --[{rel.label}]--> {tgt}")
        print(f"    {rel.properties.get('relationship_description', '')}")


inspect_extraction(0)


=== Title ===
Common AI Engineer Interview Questions & Answers (2025)

=== Raw text ===
It seems like every day we hear about some new AI development—whether it's an update to popular chatbots like ChatGPT or a robot that can whip up a home-cooked meal right in your kitchen. But who are the minds behind these innovations? While there are many roles in AI, AI engineers are critical within the broader AI ecosystem. They bridge the gap between the theoretical aspects of AI research and practical, real-world applications. With AI skills in high demand—as we saw in job postings for roles like data scientist and even data analyst in 2025—the job market is fiercely competitive. One of the best ways to stand out to employers is to ace the job interview. This is your chance to show you have what it takes to succeed in this field—even without formal experience. That’s why we’ve compiled the top 10 AI engineer interview questions along with detailed sample answers, so you can be fully prepared to

---
## 11. Build Communities & Summaries

In [12]:
graph_store.build_communities()
community_summaries = graph_store.get_community_summaries()

print(f"Communities: {len(community_summaries)}")
for cid, summary in list(community_summaries.items())[:5]:
    print("=" * 100)
    print(f"Community {cid}")
    print(summary)


Running community detection...
Raw graph: 233 nodes, 400 edges
Processed graph: 189 nodes, 363 edges
Removed 4 initial isolates, 33 low-signal resources, and 3 roleless components
Found 39 communities
✅ Generated 55 community summaries
Communities: 55
Community 0
The cluster focuses on the role of an AI Engineer, who develops and integrates AI models using various algorithms and tools. Key skills include proficiency in Python and data analysis, while essential topics for interview preparation encompass AI system design, natural language processing, and experimentation. Candidates should familiarize themselves with algorithms like neural networks, gradient boosting, and reinforcement learning, as well as tools such as Keras and SHAP for model interpretation. Understanding these connections will enable candidates to effectively demonstrate their technical knowledge and project experience during interviews.
Community 1
The cluster focuses on key concepts and tools in computer vision and d

---
## 12. GraphRAG Query Engine

In [13]:
class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    llm: LLM = Field(default_factory=lambda: QUERY_LLM)

    def custom_query(self, query_str: str) -> str:
        summaries = self.graph_store.get_community_summaries()
        if not summaries:
            return "No community summaries are available yet."

        community_text = "\n\n".join(
            f"Community {cid}:\n{summary}"
            for cid, summary in summaries.items()
        )

        prompt = f'''You are answering interview-preparation questions using community briefings from a knowledge graph.

Question:
{query_str}

Community briefings:
{community_text}

Write a focused answer that:
1. Directly answers the question
2. Names the important roles, topics, skills, tools, algorithms, or resources
3. Distinguishes overlap vs role-specific preparation when relevant
4. Keeps the answer practical for someone preparing for interviews
'''

        response = self.llm.complete(prompt)
        return str(response)


query_engine = GraphRAGQueryEngine(graph_store=graph_store)
print("✅ Query engine ready")


✅ Query engine ready


---
## 13. Export Graph Data & Visualization

In [14]:
def export_graph_data(graph_store: GraphRAGStore, output_file: str = "interview_graph_data.json"):
    bundle = graph_store.get_processed_graph()
    nx_graph = bundle["graph"]
    stats = bundle["stats"]
    community_map = getattr(graph_store, "community_membership", {}) or {}

    nodes_data = []
    for node_id, attrs in nx_graph.nodes(data=True):
        nodes_data.append(
            {
                "id": node_id,
                "label": attrs.get("label", node_id),
                "type": attrs.get("type", "OTHER"),
                "description": attrs.get("description", ""),
                "degree": attrs.get("degree", nx_graph.degree(node_id)),
                "importance": attrs.get("importance", 0),
                "role_distance": attrs.get("role_distance"),
                "community": community_map.get(node_id),
                "source_count": len(attrs.get("raw_ids", [])),
            }
        )

    links_data = []
    for source, target, data in nx_graph.edges(data=True):
        links_data.append(
            {
                "source": source,
                "target": target,
                "label": data.get("relationship", ""),
                "description": data.get("description", ""),
                "weight": data.get("weight", 1),
            }
        )

    graph_data = {
        "nodes": nodes_data,
        "links": links_data,
        "communities": len(set(community_map.values())) if community_map else 0,
        "stats": stats,
    }

    pathlib.Path(output_file).write_text(json.dumps(graph_data, indent=2), encoding="utf-8")

    print(f"✅ Graph data exported to '{output_file}'")
    print(f"   Nodes:       {len(nodes_data)}")
    print(f"   Edges:       {len(links_data)}")
    print(f"   Communities: {graph_data['communities']}")
    print(
        "   Tightening:  "
        f"-{stats['removed_initial_isolates']} isolates, "
        f"-{stats['removed_leaf_resources']} leaf resources, "
        f"-{stats['removed_roleless_components']} roleless components"
    )
    return graph_data


In [15]:
HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Interview Prep GraphRAG</title>
  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <style>
    :root {
      --bg: #ece5d8;
      --bg-deep: #e0d4bf;
      --panel: rgba(255, 252, 246, 0.92);
      --ink: #182025;
      --muted: #5e676c;
      --line: rgba(24, 32, 37, 0.14);
      --shadow: 0 16px 44px rgba(24, 32, 37, 0.11);
      --accent: #0b4f6c;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      color: var(--ink);
      font-family: "Segoe UI", "Helvetica Neue", Arial, sans-serif;
      background:
        radial-gradient(circle at top left, rgba(255,255,255,0.9), transparent 28%),
        radial-gradient(circle at top right, rgba(206, 177, 119, 0.18), transparent 24%),
        linear-gradient(145deg, #faf5ec 0%, var(--bg) 52%, var(--bg-deep) 100%);
    }
    .shell {
      display: grid;
      grid-template-columns: 390px minmax(0, 1fr);
      min-height: 100vh;
    }
    aside {
      padding: 24px 22px;
      border-right: 1px solid var(--line);
      background: rgba(255,255,255,0.52);
      backdrop-filter: blur(18px);
      overflow-y: auto;
    }
    main {
      position: relative;
      min-height: 100vh;
    }
    h1 {
      margin: 0 0 8px;
      font-family: Georgia, "Times New Roman", serif;
      font-size: 2.1rem;
      line-height: 1.02;
    }
    p {
      margin: 0;
      line-height: 1.55;
      color: var(--muted);
    }
    .panel {
      margin-top: 16px;
      padding: 15px;
      border: 1px solid var(--line);
      border-radius: 18px;
      background: var(--panel);
      box-shadow: var(--shadow);
    }
    .stats {
      display: grid;
      grid-template-columns: repeat(2, minmax(0, 1fr));
      gap: 10px;
    }
    .stat {
      padding: 12px;
      border-radius: 14px;
      background: linear-gradient(180deg, rgba(11,79,108,0.1), rgba(11,79,108,0.04));
      border: 1px solid rgba(11,79,108,0.08);
    }
    .stat span {
      display: block;
      font-size: 0.82rem;
      color: var(--muted);
    }
    .stat strong {
      display: block;
      margin-top: 3px;
      font-size: 1.42rem;
    }
    .controls {
      display: grid;
      gap: 12px;
    }
    label {
      display: block;
      margin-bottom: 6px;
      font-size: 0.86rem;
      font-weight: 700;
      color: var(--ink);
    }
    input, select, button {
      width: 100%;
      padding: 11px 12px;
      border-radius: 12px;
      border: 1px solid rgba(24,32,37,0.16);
      background: #fffdf9;
      color: var(--ink);
      font-size: 0.94rem;
    }
    button {
      cursor: pointer;
      font-weight: 700;
      background: linear-gradient(180deg, rgba(11,79,108,0.13), rgba(11,79,108,0.07));
    }
    .toolbar {
      display: flex;
      gap: 10px;
    }
    .toolbar button { flex: 1; }
    .hint {
      margin-top: 10px;
      font-size: 0.9rem;
    }
    #graph {
      width: 100%;
      height: 100vh;
    }
    .legend {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
    }
    .legend-item {
      display: inline-flex;
      align-items: center;
      gap: 8px;
      padding: 7px 11px;
      border: 1px solid var(--line);
      border-radius: 999px;
      background: rgba(255,255,255,0.82);
      font-size: 0.84rem;
    }
    .swatch {
      width: 12px;
      height: 12px;
      border-radius: 999px;
    }
    .detail h2 {
      margin: 0 0 8px;
      font-size: 1.18rem;
    }
    .detail-meta {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      margin: 10px 0 12px;
    }
    .chip {
      display: inline-flex;
      align-items: center;
      padding: 5px 10px;
      border-radius: 999px;
      background: rgba(11,79,108,0.08);
      border: 1px solid rgba(11,79,108,0.12);
      font-size: 0.82rem;
    }
    .detail-group {
      margin-top: 14px;
      padding-top: 12px;
      border-top: 1px solid var(--line);
    }
    .detail-group h3 {
      display: flex;
      justify-content: space-between;
      align-items: center;
      margin: 0 0 8px;
      font-size: 0.95rem;
    }
    .detail-group ul {
      margin: 0;
      padding-left: 18px;
    }
    .detail-group li {
      margin-bottom: 6px;
      line-height: 1.45;
    }
    .tooltip {
      position: fixed;
      pointer-events: none;
      max-width: 320px;
      padding: 10px 12px;
      border-radius: 12px;
      background: rgba(18, 24, 28, 0.94);
      color: #fff;
      font-size: 0.88rem;
      line-height: 1.45;
      opacity: 0;
      transform: translate(-9999px, -9999px);
      transition: opacity 0.12s ease;
    }
    .empty {
      padding: 48px;
      color: var(--muted);
    }
    .footer-note {
      margin-top: 12px;
      font-size: 0.84rem;
      color: var(--muted);
    }
    @media (max-width: 1040px) {
      .shell { grid-template-columns: 1fr; }
      aside { border-right: 0; border-bottom: 1px solid var(--line); }
      #graph { height: 72vh; }
    }
  </style>
</head>
<body>
  <div class="shell">
    <aside>
      <h1>Interview Prep Graph</h1>
      <p>Role nodes stay anchored. The default view hides low-signal leaves so the graph reads like a product, not a raw dump.</p>

      <div class="panel stats">
        <div class="stat"><span>Nodes</span><strong id="nodeCount">-</strong></div>
        <div class="stat"><span>Edges</span><strong id="edgeCount">-</strong></div>
        <div class="stat"><span>Communities</span><strong id="communityCount">-</strong></div>
        <div class="stat"><span>Types</span><strong id="typeCount">-</strong></div>
      </div>

      <div class="panel controls">
        <div>
          <label for="searchBox">Search Nodes</label>
          <input id="searchBox" type="search" placeholder="Try: Machine Learning Engineer">
        </div>
        <div>
          <label for="typeFilter">Filter By Type</label>
          <select id="typeFilter"><option value="ALL">All node types</option></select>
        </div>
        <div>
          <label for="relationFilter">Filter By Relationship</label>
          <select id="relationFilter"><option value="ALL">All relationships</option></select>
        </div>
        <div class="toolbar">
          <button id="resetFocus">Reset Focus</button>
          <button id="toggleLeaves">Show Leaves</button>
        </div>
        <p class="hint">Click a node to isolate its direct neighborhood. Background click resets the canvas.</p>
      </div>

      <div class="panel">
        <label>Legend</label>
        <div id="legend" class="legend"></div>
      </div>

      <div class="panel detail" id="detailPanel">
        <h2>Node Details</h2>
        <p>Select a node to inspect its role, degree, community, and grouped connections.</p>
      </div>

      <div class="footer-note" id="tighteningNote"></div>
    </aside>
    <main>
      <svg id="graph"></svg>
      <div id="emptyState" class="empty" hidden>No graph data found in this HTML bundle.</div>
    </main>
  </div>
  <div id="tooltip" class="tooltip"></div>
  <script>
    const GRAPH_DATA = GRAPH_DATA_PLACEHOLDER;
    const tooltip = document.getElementById('tooltip');
    const emptyState = document.getElementById('emptyState');
    const detailPanel = document.getElementById('detailPanel');
    const searchBox = document.getElementById('searchBox');
    const typeFilter = document.getElementById('typeFilter');
    const relationFilter = document.getElementById('relationFilter');
    const resetFocusButton = document.getElementById('resetFocus');
    const toggleLeavesButton = document.getElementById('toggleLeaves');
    const tighteningNote = document.getElementById('tighteningNote');

    function showTooltip(event, html) {
      tooltip.innerHTML = html;
      tooltip.style.opacity = 1;
      tooltip.style.transform = `translate(${event.clientX + 16}px, ${event.clientY + 16}px)`;
    }

    function hideTooltip() {
      tooltip.style.opacity = 0;
      tooltip.style.transform = 'translate(-9999px, -9999px)';
    }

    function escapeHtml(value) {
      return (value || '').replace(/[&<>"]/g, char => ({
        '&': '&amp;',
        '<': '&lt;',
        '>': '&gt;',
        '"': '&quot;',
      }[char]));
    }

    function resetDetail() {
      detailPanel.innerHTML = `
        <h2>Node Details</h2>
        <p>Select a node to inspect its role, degree, community, and grouped connections.</p>
      `;
    }

    function renderDetail(node, neighbors) {
      const grouped = d3.groups(neighbors, item => item.label)
        .sort((a, b) => d3.descending(a[1].length, b[1].length) || d3.ascending(a[0], b[0]));

      const sections = grouped.map(([label, items]) => `
        <div class="detail-group">
          <h3><span>${escapeHtml(label)}</span><span>${items.length}</span></h3>
          <ul>
            ${items.slice(0, 8).map(item => `<li><strong>${escapeHtml(item.target)}</strong>${item.description ? ` — ${escapeHtml(item.description)}` : ''}</li>`).join('')}
          </ul>
        </div>
      `).join('');

      const communityText = Number.isInteger(node.community) ? `Community ${node.community}` : 'Unclustered';
      detailPanel.innerHTML = `
        <h2>${escapeHtml(node.label)}</h2>
        <div class="detail-meta">
          <span class="chip">${escapeHtml(node.type)}</span>
          <span class="chip">Degree ${node.degree ?? 0}</span>
          <span class="chip">${escapeHtml(communityText)}</span>
        </div>
        <p>${escapeHtml(node.description || 'No description available.')}</p>
        ${sections || '<div class="detail-group"><ul><li>No direct connections are visible for this node.</li></ul></div>'}
      `;
    }

    function renderGraph(data) {
      if (!data || !data.nodes || !data.nodes.length) {
        emptyState.hidden = false;
        return;
      }

      document.getElementById('nodeCount').textContent = data.nodes.length;
      document.getElementById('edgeCount').textContent = data.links.length;
      document.getElementById('communityCount').textContent = data.communities || 0;

      const nodeTypes = [...new Set(data.nodes.map(node => node.type))].sort();
      const relationTypes = [...new Set(data.links.map(link => link.label).filter(Boolean))].sort();
      document.getElementById('typeCount').textContent = nodeTypes.length;
      typeFilter.innerHTML = '<option value="ALL">All node types</option>';
      relationFilter.innerHTML = '<option value="ALL">All relationships</option>';

      const color = d3.scaleOrdinal()
        .domain(nodeTypes)
        .range(['#0b4f6c', '#d17a22', '#0f8b8d', '#9a5f80', '#be4b49', '#6e6a5e', '#6f8796']);

      const legend = document.getElementById('legend');
      legend.innerHTML = '';

      nodeTypes.forEach(type => {
        const option = document.createElement('option');
        option.value = type;
        option.textContent = type;
        typeFilter.appendChild(option);

        const item = document.createElement('div');
        item.className = 'legend-item';
        item.innerHTML = `<span class="swatch" style="background:${color(type)}"></span>${type}`;
        legend.appendChild(item);
      });

      relationTypes.forEach(type => {
        const option = document.createElement('option');
        option.value = type;
        option.textContent = type;
        relationFilter.appendChild(option);
      });

      if (data.stats) {
        tighteningNote.textContent =
          `Tightened from ${data.stats.raw_nodes} raw nodes / ${data.stats.raw_edges} raw edges ` +
          `to ${data.stats.processed_nodes} nodes / ${data.stats.processed_edges} edges.`;
      }

      const nodeById = new Map(data.nodes.map(node => [node.id, node]));
      const roleNodes = data.nodes.filter(node => node.type === 'JOB_ROLE');
      const roleIndex = new Map(roleNodes.map((node, index) => [node.id, index]));
      const defaultVisibleIds = new Set(
        data.nodes
          .filter(node => node.type === 'JOB_ROLE' || (node.degree || 0) >= 2)
          .map(node => node.id)
      );

      let showLeaves = false;
      let activeNodeId = null;

      const svg = d3.select('#graph');
      svg.selectAll('*').remove();
      const width = svg.node().clientWidth;
      const height = svg.node().clientHeight;

      const roleRingRadius = Math.min(width, height) * 0.33;
      roleNodes.forEach((node, index) => {
        const angle = (Math.PI * 2 * index) / Math.max(1, roleNodes.length) - Math.PI / 2;
        node.x = width / 2 + Math.cos(angle) * roleRingRadius;
        node.y = height / 2 + Math.sin(angle) * roleRingRadius;
      });
      data.nodes.filter(node => node.type !== 'JOB_ROLE').forEach(node => {
        node.x = width / 2 + (Math.random() - 0.5) * 180;
        node.y = height / 2 + (Math.random() - 0.5) * 180;
      });

      const container = svg.append('g');
      svg.call(
        d3.zoom()
          .scaleExtent([0.42, 3.2])
          .on('zoom', event => container.attr('transform', event.transform))
      );

      const linkLayer = container.append('g');
      const nodeLayer = container.append('g');
      const labelLayer = container.append('g');

      function nodeRadius(node) {
        if (node.type === 'JOB_ROLE') {
          return 25 + Math.min(7, Math.sqrt(node.degree || 0));
        }
        const base = {
          INTERVIEW_TOPIC: 10,
          SKILL: 9,
          TOOL: 8.5,
          ALGORITHM: 8.5,
          RESOURCE: 7.5,
        }[node.type] || 8;
        return base + Math.min(3.8, Math.sqrt(node.degree || 0));
      }

      function relationPass(link) {
        return relationFilter.value === 'ALL' || link.label === relationFilter.value;
      }

      function linkEndpoints(link) {
        return {
          source: link.source.id || link.source,
          target: link.target.id || link.target,
        };
      }

      function currentAdjacency() {
        const adjacency = new Map(data.nodes.map(node => [node.id, new Set([node.id])]));
        data.links.forEach(link => {
          if (!relationPass(link)) return;
          const { source, target } = linkEndpoints(link);
          adjacency.get(source)?.add(target);
          adjacency.get(target)?.add(source);
        });
        return adjacency;
      }

      const links = linkLayer
        .selectAll('line')
        .data(data.links)
        .enter()
        .append('line')
        .attr('stroke', 'rgba(24, 32, 37, 0.2)')
        .attr('stroke-width', d => 1 + Math.min(2, d.weight || 1))
        .attr('stroke-dasharray', d => d.label === 'RECOMMENDED_FOR' ? '6 5' : (d.label === 'RELATED_TO' ? '3 4' : null));

      const nodes = nodeLayer
        .selectAll('circle')
        .data(data.nodes)
        .enter()
        .append('circle')
        .attr('r', d => nodeRadius(d))
        .attr('fill', d => color(d.type))
        .attr('stroke', '#fffdf7')
        .attr('stroke-width', d => d.type === 'JOB_ROLE' ? 3.4 : 1.9)
        .call(
          d3.drag()
            .on('start', (event, d) => {
              if (!event.active) simulation.alphaTarget(0.24).restart();
              d.fx = d.x;
              d.fy = d.y;
            })
            .on('drag', (event, d) => {
              d.fx = event.x;
              d.fy = event.y;
            })
            .on('end', (event, d) => {
              if (!event.active) simulation.alphaTarget(0);
              d.fx = null;
              d.fy = null;
            })
        );

      const labels = labelLayer
        .selectAll('text')
        .data(data.nodes)
        .enter()
        .append('text')
        .text(d => d.label)
        .attr('font-size', d => d.type === 'JOB_ROLE' ? 14 : 10.8)
        .attr('font-weight', d => d.type === 'JOB_ROLE' ? 700 : 500)
        .attr('fill', '#22313a')
        .attr('dx', d => nodeRadius(d) + 6)
        .attr('dy', 4);

      const simulation = d3.forceSimulation(data.nodes)
        .force('link', d3.forceLink(data.links).id(d => d.id).distance(d => {
          const sourceType = (d.source.type || '').toString();
          const targetType = (d.target.type || '').toString();
          return sourceType === 'JOB_ROLE' || targetType === 'JOB_ROLE' ? 215 : 138;
        }).strength(d => {
          const sourceType = (d.source.type || '').toString();
          const targetType = (d.target.type || '').toString();
          return sourceType === 'JOB_ROLE' || targetType === 'JOB_ROLE' ? 0.2 : 0.11;
        }))
        .force('charge', d3.forceManyBody().strength(d => d.type === 'JOB_ROLE' ? -1700 : -620))
        .force('center', d3.forceCenter(width / 2, height / 2))
        .force('roleX', d3.forceX(d => {
          if (d.type !== 'JOB_ROLE') return width / 2;
          const index = roleIndex.get(d.id) || 0;
          const angle = (Math.PI * 2 * index) / Math.max(1, roleNodes.length) - Math.PI / 2;
          return width / 2 + Math.cos(angle) * roleRingRadius;
        }).strength(d => d.type === 'JOB_ROLE' ? 0.3 : 0.03))
        .force('roleY', d3.forceY(d => {
          if (d.type !== 'JOB_ROLE') return height / 2;
          const index = roleIndex.get(d.id) || 0;
          const angle = (Math.PI * 2 * index) / Math.max(1, roleNodes.length) - Math.PI / 2;
          return height / 2 + Math.sin(angle) * roleRingRadius;
        }).strength(d => d.type === 'JOB_ROLE' ? 0.3 : 0.03))
        .force('collision', d3.forceCollide().radius(d => nodeRadius(d) + (d.type === 'JOB_ROLE' ? 18 : 12)));

      simulation.on('tick', () => {
        links
          .attr('x1', d => d.source.x)
          .attr('y1', d => d.source.y)
          .attr('x2', d => d.target.x)
          .attr('y2', d => d.target.y);

        nodes
          .attr('cx', d => d.x)
          .attr('cy', d => d.y);

        labels
          .attr('x', d => d.x)
          .attr('y', d => d.y);
      });

      function visibleNodeSet() {
        const term = searchBox.value.trim().toLowerCase();
        const selectedType = typeFilter.value;
        const adjacency = currentAdjacency();

        let baseVisible = data.nodes.filter(node => {
          if (selectedType !== 'ALL' && node.type !== selectedType) return false;
          if (!showLeaves && !term && node.type !== 'JOB_ROLE' && !defaultVisibleIds.has(node.id)) return false;
          if (term && !node.label.toLowerCase().includes(term)) return false;
          if (relationFilter.value !== 'ALL' && !(adjacency.get(node.id) && adjacency.get(node.id).size > 1) && node.type !== 'JOB_ROLE') return false;
          return true;
        }).map(node => node.id);

        const visible = new Set(baseVisible);
        data.nodes.filter(node => node.type === 'JOB_ROLE').forEach(node => visible.add(node.id));

        if (term) {
          [...visible].forEach(nodeId => {
            (adjacency.get(nodeId) || new Set()).forEach(neighborId => visible.add(neighborId));
          });
        }

        if (!activeNodeId) {
          return { visible, adjacency };
        }

        const focused = adjacency.get(activeNodeId) || new Set([activeNodeId]);
        const filtered = new Set([...focused].filter(nodeId => visible.has(nodeId)));
        filtered.add(activeNodeId);
        data.nodes.filter(node => node.type === 'JOB_ROLE').forEach(node => filtered.add(node.id));
        return { visible: filtered, adjacency };
      }

      function applyVisibility() {
        const { visible, adjacency } = visibleNodeSet();
        const focused = activeNodeId ? (adjacency.get(activeNodeId) || new Set([activeNodeId])) : null;

        nodes
          .style('display', d => visible.has(d.id) || d.type === 'JOB_ROLE' ? null : 'none')
          .style('opacity', d => {
            if (!activeNodeId) return d.type === 'JOB_ROLE' ? 1 : 0.94;
            if (focused.has(d.id)) return 1;
            return d.type === 'JOB_ROLE' ? 0.72 : 0.06;
          })
          .attr('stroke', d => d.id === activeNodeId ? '#182025' : '#fffdf7')
          .attr('stroke-width', d => d.id === activeNodeId ? 4.2 : (d.type === 'JOB_ROLE' ? 3.4 : 1.9));

        labels
          .style('display', d => {
            if (!(visible.has(d.id) || d.type === 'JOB_ROLE')) return 'none';
            if (activeNodeId) return null;
            if (d.type === 'JOB_ROLE') return null;
            if (searchBox.value.trim()) return null;
            return (d.degree || 0) >= 4 ? null : 'none';
          })
          .style('opacity', d => {
            if (!activeNodeId) return d.type === 'JOB_ROLE' ? 1 : 0.86;
            if (focused.has(d.id)) return 1;
            return d.type === 'JOB_ROLE' ? 0.86 : 0.04;
          });

        links
          .style('display', d => {
            if (!relationPass(d)) return 'none';
            const { source, target } = linkEndpoints(d);
            if (!visible.has(source) || !visible.has(target)) return 'none';
            if (!activeNodeId) return null;
            return source === activeNodeId || target === activeNodeId ? null : 'none';
          })
          .style('opacity', d => {
            if (!relationPass(d)) return 0;
            if (!activeNodeId) return d.label === 'RECOMMENDED_FOR' ? 0.25 : 0.5;
            const { source, target } = linkEndpoints(d);
            return source === activeNodeId || target === activeNodeId ? 0.96 : 0.05;
          })
          .attr('stroke-width', d => {
            const width = 1 + Math.min(2.4, d.weight || 1);
            if (!activeNodeId) return width;
            const { source, target } = linkEndpoints(d);
            return source === activeNodeId || target === activeNodeId ? width + 1 : width;
          });
      }

      function focusNode(node) {
        activeNodeId = node.id;
        const neighbors = data.links
          .filter(link => relationPass(link))
          .filter(link => {
            const { source, target } = linkEndpoints(link);
            return source === node.id || target === node.id;
          })
          .map(link => {
            const { source, target } = linkEndpoints(link);
            return {
              label: link.label,
              target: source === node.id ? target : source,
              description: link.description || '',
            };
          });
        renderDetail(node, neighbors);
        applyVisibility();
      }

      function clearFocus() {
        activeNodeId = null;
        resetDetail();
        applyVisibility();
      }

      nodes
        .on('mouseover', (event, d) => showTooltip(event, `<strong>${escapeHtml(d.label)}</strong><br>${escapeHtml(d.type)}<br>Degree: ${d.degree || 0}`))
        .on('mousemove', (event, d) => showTooltip(event, `<strong>${escapeHtml(d.label)}</strong><br>${escapeHtml(d.type)}<br>Degree: ${d.degree || 0}`))
        .on('mouseout', hideTooltip)
        .on('click', (event, d) => {
          event.stopPropagation();
          focusNode(d);
        });

      svg.on('click', () => clearFocus());

      searchBox.addEventListener('input', applyVisibility);
      typeFilter.addEventListener('change', applyVisibility);
      relationFilter.addEventListener('change', applyVisibility);
      resetFocusButton.addEventListener('click', () => clearFocus());
      toggleLeavesButton.addEventListener('click', () => {
        showLeaves = !showLeaves;
        toggleLeavesButton.textContent = showLeaves ? 'Hide Leaves' : 'Show Leaves';
        applyVisibility();
      });

      resetDetail();
      applyVisibility();
    }

    renderGraph(GRAPH_DATA);
  </script>
</body>
</html>'''


def generate_graph_html(
    graph_data: dict,
    output_file: str = GRAPH_OUTPUT_FILE,
):
    html = HTML_TEMPLATE.replace("GRAPH_DATA_PLACEHOLDER", json.dumps(graph_data).replace("</", "<\/"))
    pathlib.Path(output_file).write_text(html, encoding="utf-8")
    print(f"✅ Visualization saved to '{output_file}'")
    print("   Open it in your browser to explore the role-centered graph.")


graph_data = export_graph_data(graph_store=graph_store)
generate_graph_html(graph_data=graph_data)


✅ Graph data exported to 'interview_graph_data.json'
   Nodes:       189
   Edges:       363
   Communities: 39
   Tightening:  -4 isolates, -33 leaf resources, -3 roleless components
✅ Visualization saved to 'job_skill_graph.html'
   Open it in your browser to explore the role-centered graph.


---
## 14. Example Queries

In [16]:
q1 = "What should I study for Machine Learning Engineer interviews?"
print(f"Query: {q1}")
print("=" * 80)
print(query_engine.custom_query(q1))


Query: What should I study for Machine Learning Engineer interviews?
To prepare for Machine Learning Engineer interviews, focus on both foundational and advanced topics, skills, and tools that are crucial for the role. Here’s a structured approach:

1. **Core Algorithms and Concepts**:
   - Study key algorithms such as Logistic Regression, Random Forest, Naive Bayes, K-means, Decision Trees, and XGBoost. Understanding these will help you tackle questions related to model selection and evaluation.
   - Grasp the principles of Gradient Descent and its variants, as they are fundamental to optimizing machine learning models.

2. **Model Evaluation and System Design**:
   - Familiarize yourself with Model Evaluation techniques and metrics, such as Cross-Validation, to assess model performance.
   - Understand Machine Learning System Design, which involves integrating models into scalable systems.

3. **Feature Engineering**:
   - Develop skills in Feature Engineering to transform raw data i

In [17]:
q2 = "What overlaps between Machine Learning Engineer and Data Scientist interview preparation?"
print(f"Query: {q2}")
print("=" * 80)
print(query_engine.custom_query(q2))


Query: What overlaps between Machine Learning Engineer and Data Scientist interview preparation?
When preparing for interviews as a Machine Learning Engineer or Data Scientist, there are several overlapping areas, as well as role-specific topics to consider.

**Overlapping Areas:**

1. **Algorithms and Techniques:**
   - Both roles require a strong understanding of algorithms such as Logistic Regression, Random Forest, and XGBoost. These are fundamental for building predictive models and are commonly discussed in interviews (Community 3, 20, 23).

2. **Statistical Methods:**
   - Proficiency in statistical methods like Hypothesis Testing, A/B Testing, and understanding the Central Limit Theorem is crucial for both roles. These methods are essential for data interpretation and decision-making (Community 19, 44, 54).

3. **Model Evaluation and Cross-Validation:**
   - Both roles emphasize the importance of Model Evaluation and Cross-Validation to ensure model performance and generalizati

---
## 15. Summary

This project now follows the reference-notebook shape:

- **Scrape notebook**: SerpAPI search → article scraping → CSV export
- **Graph notebook**: LlamaIndex `PropertyGraphIndex` → custom extractor → custom graph store → community summaries → HTML export

The graph is designed to answer practical interview-prep questions rather than generic labor-market questions.